In [1]:
!pip3 install transformers datasets torch scikit-learn

In [2]:
#import libraries
from transformers import BertTokenizer, BertForSequenceClassification , Trainer, TrainingArguments
from datasets import load_dataset
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [3]:
#load dataset
dataset = load_dataset("imdb")
dataset = dataset.shuffle(seed=42)
small_train = dataset["train"].select(range(500))
small_test = dataset["test"].select(range(200))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
#Tokenize the dataset
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)
train_enc = small_train.map(tokenize_function, batched=True)
test_enc = small_test.map(tokenize_function, batched=True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [5]:
#prepare the pytorch
train_enc.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_enc.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [6]:
#loading the pretrained model for classification
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
#define the evaluation metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    labels = labels.astype(int)
    preds = preds.astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": float(acc),
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall)
    }

In [8]:
!pip install -U accelerate
!pip install -U transformers

In [9]:
import accelerate
print(accelerate.__version__)

1.13.0


In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=".results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=2,
)

In [11]:
#initilize the trainer
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_enc,
    eval_dataset=test_enc,
    compute_metrics=compute_metrics
)

In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,No log,0.359203,0.860000,0.847826,0.886364,0.812500
2,No log,0.320034,0.885000,0.879581,0.884211,0.875000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=126, training_loss=0.41701783074273, metrics={'train_runtime': 133.1126, 'train_samples_per_second': 7.512, 'train_steps_per_second': 0.947, 'total_flos': 263111055360000.0, 'train_loss': 0.41701783074273, 'epoch': 2.0})

In [26]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision,Recall
No log,0.320034,2,0.885000,0.879581,0.884211,0.875000


{'eval_loss': 0.32003429532051086,
 'eval_accuracy': 0.885,
 'eval_f1': 0.8795811518324608,
 'eval_precision': 0.8842105263157894,
 'eval_recall': 0.875}

In [39]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

text_to_predict = 'I like this product'

inputs = tokenizer(text_to_predict, return_tensors="pt").to(device)
outputs = model(**inputs)

prediction = torch.argmax(outputs.logits).item()

print(f"Text: '{text_to_predict}'")
print('Predicted Sentiments', 'Positive' if prediction == 1 else 'Negative')

Text: 'I like this product'
Predicted Sentiments Positive
